# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to explore and analyze the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library. You will load dataset metadata, inspect record sets, extract data, perform exploratory data analysis, and visualize results, all driven by the dataset's Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL (Croissant schema)
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, their `@id`s and relationships as defined in the Croissant schema.

In [ ]:
# List available record sets and their fields using the metadata structure
print("Record Sets available in this dataset:\n")
record_sets = []
if hasattr(metadata, 'record_sets'):
    for rs in metadata.record_sets:
        print(f"- @id: {rs['@id']}  name: {rs.get('name', '(no name)')}")
        print("  Fields:")
        for f in rs.get('fields', []):
            print(f"    - @id: {f['@id']}   name: {f.get('name', f['@id'])}   dataType: {f.get('dataType', '?')}")
        record_sets.append(rs['@id'])
else:
    # Fallback: try to find 'recordSet' attribute directly as per possible dictionary structure
    rec_sets = getattr(metadata, 'recordSet', None)
    if isinstance(rec_sets, list):
        for rs in rec_sets:
            if isinstance(rs, dict):
                print(f"- @id: {rs.get('@id', '')}  name: {rs.get('name', '(no name)')}")
                print("  Fields:")
                for f in rs.get('field', []):
                    print(f"    - @id: {f['@id']}   name: {f.get('name', f['@id'])}   dataType: {f.get('dataType', '?')}")
                record_sets.append(rs['@id'])
    elif isinstance(rec_sets, dict):
        rs = rec_sets
        print(f"- @id: {rs.get('@id', '')}  name: {rs.get('name', '(no name)')}")
        print("  Fields:")
        for f in rs.get('field', []):
            print(f"    - @id: {f['@id']}   name: {f.get('name', f['@id'])}   dataType: {f.get('dataType', '?')}")
        record_sets.append(rs['@id'])
else:
    print("No record sets found in metadata.")

# If record_sets are still empty, try using the recommended API (mlcroissant >=0.5.2)
if not record_sets:
    if hasattr(dataset, 'list_record_sets'):
        for rs in dataset.list_record_sets():
            print(f"- @id: {rs['@id']}  name: {rs.get('name', '')}")
            record_sets.append(rs['@id'])

## 3. Data Extraction
Load data from each discovered record set into a DataFrame for quick analysis. All references use the `@id` field. If you know the desired record set `@id` (e.g., from above), you may specify it directly.

In [ ]:
# Get record set IDs (by @id) either from the above block or via dataset API.
# If record_sets is empty, attempt to fetch names/ids using dataset.list_record_sets().
if not record_sets:
    # fallback
    rs_infos = getattr(dataset, 'list_record_sets', lambda: [])()
    record_sets = [rs['@id'] for rs in rs_infos] if rs_infos else []

if not record_sets:
    raise RuntimeError("Could not discover any record sets in dataset metadata.")

print(f"Discovered record sets: {record_sets}")
dataframes = {}

for record_set_id in record_sets:
    print(f"Loading records from {record_set_id} ...")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"\nColumns in {record_set_id}: {dataframes[record_set_id].columns.tolist()}")
        display(dataframes[record_set_id].head())
    else:
        print(f"No records found for record set {record_set_id}.")

# Choose the main survey record set for the following analysis
main_record_set = record_sets[0]
# Show available columns in the main DataFrame.
print(f"Columns in main record set ({main_record_set}):\n", dataframes[main_record_set].columns.tolist())

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps like filtering, normalization, and grouping. All field references are by `@id`. Choose a numeric field and a grouping field from your columns above.

In [ ]:
# Select a numeric field (by @id) for filtering/normalizing; adjust as found above.
# Example: Suppose 'phq9_total_score' and 'level_of_education' are field @ids as per this dataset.
numeric_field_id = 'phq9_total_score'  # replace with the actual @id from your DataFrame
group_field_id = 'level_of_education'  # replace as appropriate

df = dataframes[main_record_set]

# Check that selected fields exist
if numeric_field_id not in df.columns:
    print(f"Field {numeric_field_id} not found in columns: {df.columns.tolist()}")
else:
    # Remove missing/invalid entries
    filtered_df = df[df[numeric_field_id].notnull()].copy()
    # Filter for high scores as example
    threshold = 10
    filtered_df = filtered_df[filtered_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold} (n={len(filtered_df)}):")
    display(filtered_df.head())

    # Normalize the selected numeric field
    col_norm = f"{numeric_field_id}_normalized"
    filtered_df[col_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, col_norm]].head())

    # Group by a categorical field if present
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
        display(grouped_df)
    else:
        print(f"Group field {group_field_id} not found in DataFrame columns.")

## 5. Visualization
Visualize distributions and relationships between fields using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram for the PHQ-9 total score
if numeric_field_id in df.columns:
    plt.figure(figsize=(7, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.xlabel('PHQ-9 Total Score')
    plt.ylabel('Count')
    plt.title('Distribution of PHQ-9 Total Scores')
    plt.show()

# Boxplot by education level, if the group field exists
if group_field_id in df.columns and numeric_field_id in df.columns:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
    plt.xlabel('Level of Education')
    plt.ylabel('PHQ-9 Total Score')
    plt.title('PHQ-9 Total Score by Level of Education')
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR² Mental Health Survey Dataset using `mlcroissant`.
- We inspected dataset metadata and discovered available record sets and fields via their `@id`s.
- We loaded and visualized survey records, normalized scores, and grouped statistics to understand mental health patterns.
- By referencing fields and record sets by `@id`, you can programmatically process complex FAIR data packages.

This workflow can be adapted for other Croissant-compliant datasets with multiple record sets and fields.